# Graded Assignment 1 — run everything on Colab

Runtime → Change runtime type → **T4 GPU** (or better), then Runtime → Run all.

* `checkpoints/` and `results/` live on Google Drive (`MyDrive/ee58012-ga1/`), so a
  disconnect loses nothing: re-run all cells and finished steps are skipped
  (training is skipped if the checkpoint exists, ASR tables resume per configuration).
* Already have the Self-Study 2 model? Put `resnet18_clean_seed0.pt` in
  `MyDrive/ee58012-ga1/checkpoints/` and training is skipped.
* Private repository: add a Colab secret `GITHUB_TOKEN` (key icon on the left) with a
  GitHub token that can read the repository.

In [ ]:
REPO_URL = "https://github.com/<user>/<repo>.git"   # <- set this
DRIVE_DIR = "/content/drive/MyDrive/ee58012-ga1"
WORK = "/content/ga1"

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")

url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass                                   # public repository, no token needed
if os.path.isdir(WORK):
    subprocess.run(["git", "-C", WORK, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", url, WORK], check=True)

for sub in ("checkpoints", "results"):     # keep outputs on Drive
    os.makedirs(f"{DRIVE_DIR}/{sub}", exist_ok=True)
    local = f"{WORK}/{sub}"
    if not os.path.islink(local):
        subprocess.run(["rm", "-rf", local], check=True)
        os.symlink(f"{DRIVE_DIR}/{sub}", local)
os.chdir(WORK)
!git log -1 --oneline

## 1. Clean ResNet-18 (Self-Study 2 recipe, ~1–1.5 h on a T4)

In [ ]:
CKPT = "checkpoints/resnet18_clean_seed0.pt"
if os.path.exists(CKPT):
    print("checkpoint found, training skipped:", os.path.realpath(CKPT))
else:
    !python -W ignore main.py --optimizer momentum --scheduler cosine --lr 0.1 --epochs 100 --workers 2
!python -W ignore main.py --mode test

## 2. Part A, Task 1 — Grad-CAM maps and masks for 5 test images

In [ ]:
!python -W ignore -m experiments.figures --figure gradcam_masks --batch-size 500
from IPython.display import Image, display
display(Image("results/fig_gradcam_masks.png", width=700))

## 3. Part A, Task 2 + Part B, Task 2 — ASR table (all correctly classified test images)

In [ ]:
!python -W ignore -m experiments.run_attacks --batch-size 500 --out results/asr

## 4. Part A, Task 3 — ablation: which Grad-CAM score for the targeted mask

In [ ]:
for score in ("true", "target", "margin"):
    !python -W ignore -m experiments.run_attacks --batch-size 500 --modes targeted --methods gradcam --cam-score {score} --out results/asr_cam_{score}

## 5. Part B, Task 3 — Grad-CAM mask vs learned masks for 3 images

In [ ]:
!python -W ignore -m experiments.figures --figure learned_vs_gradcam --rho 0.25
display(Image("results/fig_learned_vs_gradcam.png", width=900))

## 6. Summary

In [ ]:
for f in ["results/asr.md"] + [f"results/asr_cam_{s}.md" for s in ("true", "target", "margin")]:
    print(f"== {f}"); print(open(f).read())
!ls -la results/ checkpoints/